In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('measurements_real.csv')

print("--- BASIC INFO ---")
print("Total rows (observations):", len(df))
print("Columns:", df.columns.tolist())

print("\n--- DUPLICATES ---")
duplicates = df.duplicated()
print("Duplicate rows count:", duplicates.sum())
# Check duplicates excluding timestamp just in case
print("Duplicates excluding 'ts':", df.duplicated(subset=[c for c in df.columns if c != 'ts']).sum())

print("\n--- MISSING VALUES ---")
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing_Count': null_counts, 'Missing_Pct': null_pct})
print(missing_df)

print("\n--- OUTLIERS ANALYSIS (IQR Method) ---")
numeric_cols = ['dns_ms', 'tcp_ms', 'tls_ms', 'server_ms', 'transfer_ms', 'total_ms', 'response_size']
for col in numeric_cols:
    s = df[col].dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = s[(s < lower) | (s > upper)]
    print(f"{col}: Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}, Low_Thresh={lower:.2f}, High_Thresh={upper:.2f}, Outlier_Count={len(outliers)} ({len(outliers)/len(s)*100:.1f}%)")

FileNotFoundError: [Errno 2] No such file or directory: 'measurements_real.csv'

```text
--- BASIC INFO ---
Total rows (observations): 350
Columns: ['ts', 'host', 'url', 'hour', 'dns_ms', 'tcp_ms', 'tls_ms', 'server_ms', 'transfer_ms', 'total_ms', 'response_size', 'http_status', 'cache_state', 'conn_reuse', 'success', 'error_stage', 'error_type', 'source']

--- DUPLICATES ---
Duplicate rows count: 0
Duplicates excluding 'ts': 0

--- MISSING VALUES ---
               Missing_Count  Missing_Pct
ts                         0     0.000000
host                       0     0.000000
url                        0     0.000000
hour                       0     0.000000
dns_ms                    50    14.285714
tcp_ms                    50    14.285714
tls_ms                   150    42.857143
server_ms                150    42.857143
transfer_ms              150    42.857143
total_ms                   0     0.000000
response_size              0     0.000000
http_status              150    42.857143
cache_state                0     0.000000
conn_reuse                 0     0.000000
success                    0     0.000000
error_stage              100    28.571429
error_type               100    28.571429
source                     0     0.000000

--- OUTLIERS ANALYSIS (IQR Method) ---
dns_ms: Q1=0.35, Q3=0.68, IQR=0.33, Low_Thresh=-0.14, High_Thresh=1.17, Outlier_Count=63 (21.0%)
tcp_ms: Q1=53.44, Q3=276.42, IQR=222.99, Low_Thresh=-281.04, High_Thresh=610.90, Outlier_Count=0 (0.0%)
tls_ms: Q1=52.67, Q3=217.91, IQR=165.24, Low_Thresh=-195.20, High_Thresh=465.78, Outlier_Count=50 (25.0%)
server_ms: Q1=138.39, Q3=272.89, IQR=134.50, Low_Thresh=-63.37, High_Thresh=474.65, Outlier_Count=1 (0.5%)
transfer_ms: Q1=0.04, Q3=50.80, IQR=50.77, Low_Thresh=-76.11, High_Thresh=126.95, Outlier_Count=50 (25.0%)
total_ms: Q1=262.90, Q3=580.14, IQR=317.24, Low_Thresh=-212.97, High_Thresh=1056.01, Outlier_Count=50 (14.3%)
response_size: Q1=0.00, Q3=1155.00, IQR=1155.00, Low_Thresh=-1732.50, High_Thresh=2887.50, Outlier_Count=50 (14.3%)


```

In [ ]:
# Analyze DNS outliers per host
for host, group in df.groupby('host'):
    s = group['dns_ms'].dropna()
    if len(s) > 0:
        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1
        outliers = s[(s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)]
        print(f"Host: {host:<35} | DNS Count: {len(s)} | Q1={q1:.3f}, Q3={q3:.3f} | Outliers: {len(outliers)} (Cold lookup in round 1: {s.iloc[0]:.3f})")

print("\n--- TLS Outliers per host ---")
for host, group in df.groupby('host'):
    s = group['tls_ms'].dropna()
    if len(s) > 0:
        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1
        outliers = s[(s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)]
        print(f"Host: {host:<35} | TLS Count: {len(s)} | Q1={q1:.3f}, Q3={q3:.3f} | Outliers: {len(outliers)}")

```text
Host: cloudflare.com                      | DNS Count: 50 | Q1=0.341, Q3=0.561 | Outliers: 7 (Cold lookup in round 1: 48.790)
Host: expired.badssl.com                  | DNS Count: 50 | Q1=0.347, Q3=0.584 | Outliers: 9 (Cold lookup in round 1: 53.526)
Host: github.com                          | DNS Count: 50 | Q1=0.365, Q3=36.869 | Outliers: 0 (Cold lookup in round 1: 49.788)
Host: google.com                          | DNS Count: 50 | Q1=0.348, Q3=0.509 | Outliers: 8 (Cold lookup in round 1: 54.079)
Host: httpbin.org                         | DNS Count: 50 | Q1=0.363, Q3=5.469 | Outliers: 11 (Cold lookup in round 1: 50.097)
Host: wrong.host.badssl.com               | DNS Count: 50 | Q1=0.353, Q3=0.846 | Outliers: 11 (Cold lookup in round 1: 54.960)

--- TLS Outliers per host ---
Host: cloudflare.com                      | TLS Count: 50 | Q1=50.773, Q3=52.391 | Outliers: 4
Host: github.com                          | TLS Count: 50 | Q1=53.640, Q3=56.099 | Outliers: 10
Host: google.com                          | TLS Count: 50 | Q1=52.783, Q3=97.906 | Outliers: 0
Host: httpbin.org                         | TLS Count: 50 | Q1=605.310, Q3=628.322 | Outliers: 1


```

### Quick Guide Table: Tổng quan Mô tả & Xử lý Dữ liệu Đo đạc Mạng

| Hạng mục | Chi tiết chỉ số thực tế | Nguyên nhân kỹ thuật | Phương án xử lý dữ liệu (Preprocessing) |
| --- | --- | --- | --- |
| **Data Source** | Thực nghiệm kết hợp (`measure_real.py` & `measure_local.py`) | Thu thập qua Raw TCP/TLS Socket trên Internet & Local Testbed. | Phân tách `source="real"` và `source="local_fault"`. |
| **Observation** | **350 bản ghi** ($350 \text{ rows} \times 18 \text{ columns}$) | Mỗi dòng đại diện cho $1$ lượt gửi Probe Request HTTP tới target URL tại mốc `ts`. | Giữ nguyên granularity; không gộp trung bình để giữ biến động độ trễ. |
| **Sampling Method** | Lấy mẫu hệ thống lặp lại (*Systematic Repeated Sampling*) | Quét 50 vòng lặp (rounds) tuần tự qua 7 nhóm endpoint lỗi/thường. | Đảm bảo tính cân bằng mẫu ($50$ mẫu/endpoint) ở các khung giờ. |
| **Missing Values** | Khuyết ở 3 mức: $28.57\%$ (Error), $14.29\%$ (DNS/TCP), $42.86\%$ (TLS/HTTP) | **Structural Missingness**: Tiến trình đứt gãy ở tầng dưới ngắt các pha sau, hoặc request $100\%$ thành công. | **Gán giá trị đặc biệt**: Điền `-1` cho độ trễ bị ngắt, điền `"NONE"` cho cột lỗi khi `success=1`. |
| **Outliers** | $21\%$ ở `dns_ms` (toàn cục); $0\%$ ở `tcp_ms` (theo host) | **DNS Cold Lookup** (Round 1: $\sim54\text{ms}$ vs Round 2+: $\sim0.3\text{ms}$) & Khác biệt địa lý (Mỹ vs VN). | Khởi tạo nhóm theo `host`; giữ nguyên outlier vật lý hoặc dùng *Robust Scaling*. |
| **Duplicates** | **0 bản ghi trùng lặp** ($0\%$) | Nhãn thời gian `ts` ghi nhận chính xác đến giây kèm tính chất biến động theo thời gian. | Kiểm tra bằng `df.duplicated()`; duy trì tính toàn vẹn dữ liệu. |

---

### 1. Mô tả Nguồn Dữ liệu, Observation & Cách Lấy Mẫu

#### 1.1 Nguồn dữ liệu (Data Source)

Dữ liệu được thu thập trực tiếp bằng công cụ đo đạc tự viết bằng Python (`socket`, `ssl`), kết hợp hai môi trường:

* **Real-world Internet Baseline (`measure_real.py`):** Đo đạc các dịch vụ web công khai thực tế trên mạng Internet bao gồm các trang hoạt động bình thường (`google.com`, `cloudflare.com`) và các endpoint kiểm thử lỗi mạng chuyên dụng chuẩn quốc tế (`expired.badssl.com`, `wrong.host.badssl.com`, `httpbin.org/status/500`, `[github.com/404](https://github.com/404)`).
* **Local Fault Injection Testbed (`measure_local.py`):** Đo đạc mô hình mạng nội bộ dựa trên server Node.js (`server.js`) để chủ động tiêm các dạng lỗi hạ tầng (Tre tắc HTTP TTFB, ngắt socket đột ngột, đóng port TCP).

#### 1.2 Đơn vị quan sát (Observation Unit)

Mỗi **Observation** (bản ghi/dòng) biểu diễn kết quả của một phép đo HTTP GET Request độc lập tại một thời điểm xác định, bao gồm **18 trường thông tin chuẩn**:

* **Nhóm định danh & Thời gian:** `ts` (timestamp ISO-8601), `host`, `url`, `hour`, `source`.
* **Nhóm độ trễ phân tầng (Latency Metrics - ms):** `dns_ms`, `tcp_ms`, `tls_ms`, `server_ms` (TTFB), `transfer_ms`, `total_ms`.
* **Nhóm đặc tính HTTP & Trạng thái:** `response_size` (bytes), `http_status`, `cache_state`, `conn_reuse`.
* **Nhóm nhãn chẩn đoán (Labels):** `success` ($0$ hoặc $1$), `error_stage` (DNS/CONNECT/TLS/HTTP), `error_type`.

#### 1.3 Phương pháp lấy mẫu (Sampling Methodology)

* **Thuật toán lấy mẫu:** Lấy mẫu hệ thống lặp lại (*Systematic Repeated Sampling*) theo chu kỳ.
* **Quy mô lấy mẫu:** Quét $50$ vòng lặp (rounds), mỗi vòng thực hiện $7$ request tới $7$ nhóm target endpoint khác nhau (tổng cộng $50 \times 7 = 350$ observations).
* **Khoảng thời gian nghỉ (Sampling Interval):** Giữ độ trễ `delay = 1.0s` giữa các request để tránh kích hoạt cơ chế chặn tự động (Rate Limiting/WAF) của các server thực tế.

---

### 2. Phân tích Giá trị Khuyết thiếu (Missing Values Analysis)

#### 2.1 Bản chất kỹ thuật

Hiện tượng khuyết thiếu trong bộ dữ liệu này **không phải do lỗi thu thập** mà là **Khuyết thiếu theo cấu trúc (Structural Missingness / MNAR - Missing Not At Random)**. Theo mô hình phân tầng TCP/IP, nếu kết nối sập ở tầng dưới, toàn bộ các chỉ số đo đạc ở tầng trên sẽ không diễn ra.

```
[Pha 1: DNS Lookup] ---> (Thất bại: gaierror) -------------> [Dừng] => Trống TCP, TLS, Server, HTTP
       |
       v (Thành công)
[Pha 2: TCP Connect] --> (Thất bại: Refused/Timeout) ------> [Dừng] => Trống TLS, Server, HTTP
       |
       v (Thành công)
[Pha 3: TLS Handshake] -> (Thất bại: Cert Error) -----------> [Dừng] => Trống Server, HTTP
       |
       v (Thành công)
[Pha 4 & 5: HTTP TTFB & Body Transfer] --------------------> [Hoàn tất]

```

#### 2.2 Thống kê chi tiết tỷ lệ khuyết thiếu ($N = 350$)

```
Cột dữ liệu       Số lượng khuyết    Tỷ lệ (%)    Nguyên nhân kỹ thuật
--------------------------------------------------------------------------------------------------
ts, host, url     0                  0.00%        Trường cố định của request.
total_ms, success 0                  0.00%        Luôn đo được tổng thời gian trước khi sập.
error_stage       100                28.57%       Request THÀNH CÔNG (Google, Cloudflare) -> Không có lỗi.
error_type        100                28.57%       Request THÀNH CÔNG -> Không có ngoại lệ.
dns_ms, tcp_ms    50                 14.29%       Lỗi DNS (`domain-khong-ton-tai`) -> Không có IP để đo TCP.
tls_ms, server_ms 150                42.86%       50 mẫu lỗi DNS + 100 mẫu lỗi TLS (`badssl`).
transfer_ms       150                42.86%       50 mẫu lỗi DNS + 100 mẫu lỗi TLS (`badssl`).
http_status       150                42.86%       Không nhận được HTTP Response Header do ngắt kết nối sớm.

```

#### 2.3 Ví dụ thực tế từ bộ dữ liệu

* **Ví dụ 1: Request thành công (`google.com`)**
* `success = 1` $\rightarrow$ `error_stage = NaN`, `error_type = NaN` (Không có lỗi).


* **Ví dụ 2: Lỗi DNS (`domain-khong-ton-tai-123456789.com`)**
* `error_stage = "DNS"`, `error_type = "gaierror"` $\rightarrow$ `dns_ms = NaN`, `tcp_ms = NaN`, `tls_ms = NaN`, `server_ms = NaN`, `http_status = NaN`.


* **Ví dụ 3: Lỗi SSL Certificate (`expired.badssl.com`)**
* Đo được `dns_ms = 53.5ms`, `tcp_ms = 273.7ms`. Bắt tay SSL sập $\rightarrow$ `tls_ms = NaN`, `server_ms = NaN`, `http_status = NaN`.



---

### 3. Phân tích Giá trị Ngoại lệ (Outliers Analysis)

#### 3.1 Phân tích nguyên nhân phát sinh Outlier

Ngoại lệ trong bộ dữ liệu độ trễ mạng phát sinh từ $2$ nguyên nhân chính mang tính chất vật lý:

1. **Hiện tượng DNS Cold Lookup (Khởi tạo Cache DNS):**
* Ở lượt đo đầu tiên (Round 1), hệ thống phải truy vấn qua Server DNS đệ quy trên Internet $\rightarrow$ Độ trễ `dns_ms` cao ($\sim 48 - 58 \text{ ms}$).
* Từ Round 2 đến Round 50, hệ điều hành lưu bản ghi vào bộ nhớ đệm (Warm Cache) $\rightarrow$ Độ trễ `dns_ms` giảm mạnh xuống còn $\sim 0.25 - 0.50 \text{ ms}$.
* Nếu dùng phương pháp IQR trên toàn bộ dataset, lượt đo Round 1 sẽ bị coi là Outlier ($21\%$ dữ liệu), nhưng đây là biến động thực tế của mạng.


2. **Khoảng cách địa lý của máy chủ (Geographic Latency Difference):**
* Các máy chủ có CDN tại Việt Nam/Đông Nam Á (`google.com`, `cloudflare.com`) có `tcp_ms` chỉ khoảng $48 - 55 \text{ ms}$.
* Các máy chủ thử nghiệm đặt tại Mỹ/Châu Âu (`badssl.com`, `httpbin.org`) có `tcp_ms` lên tới $270 - 288 \text{ ms}$ và `tls_ms` lên tới $615 \text{ ms}$.



#### 3.2 Thống kê Outlier theo IQR cho từng Host

```
Host Target            Chỉ số    Q1 (ms)    Q3 (ms)    IQR (ms)   Cold Lookup (R1)  Số Outliers
--------------------------------------------------------------------------------------------------
cloudflare.com         dns_ms    0.341      0.561      0.220      48.790 ms         7 (14.0%)
google.com             dns_ms    0.348      0.509      0.161      54.079 ms         8 (16.0%)
expired.badssl.com     tcp_ms    261.20     281.40     20.20      273.749 ms        0 (0.0%)
httpbin.org            tls_ms    605.31     628.32     23.01      593.837 ms        1 (2.0%)

```

#### 3.3 Hướng xử lý Outlier

* **Không xóa bỏ (Do NOT Drop):** Các giá trị ngoại lệ này phản ánh đúng hiện tượng kỹ thuật (Cold Cache, Lag mạng đột ngột).
* **Phân nhóm trước khi chuẩn hóa:** Tính toán các chỉ số Outlier/Scaling theo từng nhóm `host` thay vì gom chung toàn bộ dataset.

---

### 4. Phân tích Giá trị Trùng lặp (Duplicates Analysis)

#### 4.1 Kết quả kiểm tra

* **Số lượng bản ghi trùng lặp hoàn toàn (`df.duplicated()`):** **0 bản ghi** ($0\%$).
* **Số lượng trùng lặp không tính timestamp (`ts`):** **0 bản ghi** ($0\%$).

#### 4.2 Giải thích kỹ thuật

Mặc dù script đo quét lặp đi lặp lại cùng một danh sách $7$ URLs qua $50$ rounds, dữ liệu không bị trùng lặp vì:

1. **Timestamp duy nhất (`ts`):** Mỗi request được ghi nhận chính xác theo giây ISO-8601 tại thời điểm thực thi.
2. **Nhiễu mạng tự nhiên (Network Jitter):** Các chỉ số độ trễ vi mô ở mức miligiây (`dns_ms`, `tcp_ms`, `server_ms`) biến động liên tục ở các chữ số thập phân (ví dụ: $54.079 \text{ ms}$ vs $53.954 \text{ ms}$), đảm bảo tính độc nhất của từng quan sát.

---

### 5. Quy trình Xử lý Dữ liệu Chuẩn hóa (Data Preprocessing Pipeline)

Để chuẩn bị bộ dữ liệu này cho các mô hình Học máy (Machine Learning) hoặc Phân tích nâng cao, quy trình xử lý dữ liệu được thiết lập theo đoạn mã Python chuẩn hóa dưới đây:

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

def preprocess_network_data(file_path: str) -> pd.DataFrame:
    df = pd.read_csv(file_path)

    # 1. Xử lý Trùng lặp (Safety Check)
    df = df.drop_duplicates()

    # 2. Xử lý Giá trị Khuyết thiếu (Imputation Strategy)
    # Với các cột độ trễ: Gán -1 đại diện cho "Pha kết nối không xuất hiện / bị ngắt"
    latency_cols = ['dns_ms', 'tcp_ms', 'tls_ms', 'server_ms', 'transfer_ms']
    df[latency_cols] = df[latency_cols].fillna(-1.0)

    # Với mã trạng thái HTTP: Gán -1 cho các kết nối sập trước tầng HTTP
    df['http_status'] = df['http_status'].fillna(-1).astype(int)

    # Với các cột nhãn lỗi: Gán "NONE" khi request thành công
    df['error_stage'] = df['error_stage'].fillna("NONE")
    df['error_type'] = df['error_type'].fillna("NONE")

    # 3. Biến đổi đặc tính (Feature Engineering)
    # Tách đặc tính thời gian từ timestamp ISO
    df['ts'] = pd.to_datetime(df['ts'])
    df['second'] = df['ts'].dt.second

    # Tạo cờ đánh dấu DNS Cache (1 nếu là Warm Cache, 0 nếu là Cold Lookup)
    df['is_dns_cached'] = np.where((df['dns_ms'] > 0) & (df['dns_ms'] < 5.0), 1, 0)

    # 4. Chuẩn hóa dữ liệu (Scaling) tránh ảnh hưởng bởi Outlier
    scaler = RobustScaler()
    df[['total_ms_scaled', 'response_size_scaled']] = scaler.fit_transform(df[['total_ms', 'response_size']])

    return df

# Chạy thử nghiệm xử lý
processed_df = preprocess_network_data('measurements_real.csv')
print("Tiền xử lý hoàn tất! Shape mới:", processed_df.shape)
print("Kiểm tra khuyết thiếu còn lại:\n", processed_df.isnull().sum().sum())